In [22]:
# Bibliotecas utilizadas
import os
import pandas as pd
import pyodbc
from datetime import datetime
import warnings

# Pega usuário de rede
usuario = os.getenv('USERNAME')

# Tira mensagens de warning
warnings.filterwarnings('ignore')

# Caminho para o arquivo com credenciais
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)

# Carrega credenciais
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o SQL Server
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Query SQL
query_table = """
SELECT 
    FT.AnoMesRef,
    FT.ID_Produto,
    FT.ID_Comissionado,
    FT.ID_UF,
    FT.ID_Cota,
    FT.ST_Contrato,
    FT.Tipo_Pessoa,
    FT.Grupo,
    FT.Cota,
    FT.Faixa_Atraso,
    FT.Desclassificado,
    FT.TP_Pagto_DemaisParcelas,
    FT.ST_Adimplencia,
    DP.CD_InscricaoNacional,
    VA.DT_EntregaBem,
    CO.CANAL_VENDA,
    VB.DT_Contemplacao,

    CASE 
        WHEN VB.DT_Contemplacao IS NULL OR VA.DT_EntregaBem IS NULL THEN NULL
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 31 THEN 'Over 30'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 61 THEN 'Over 60'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 91 THEN 'Over 90'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 121 THEN 'Over 120'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 151 THEN 'Over 150'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 181 THEN 'Over 180'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 211 THEN 'Over 210'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 241 THEN 'Over 240'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 271 THEN 'Over 270'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 301 THEN 'Over 300'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 331 THEN 'Over 330'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 361 THEN 'Over 360'
        WHEN DATEDIFF(DAY, VB.DT_Contemplacao, VA.DT_EntregaBem) < 391 THEN 'Over 390'
        ELSE 'Over > 390'
    END AS Categoria_Tempo_Entrega

FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN 
    FT0002_VendaAlocacoes AS VA ON FT.ID_Cota = VA.ID_Cota
LEFT JOIN 
    DM0004_Comissionados AS CO ON FT.ID_Comissionado = CO.ID_Comissionado
LEFT JOIN 
    FT0018_Contemplacao AS VB ON FT.ID_Cota = VB.ID_Cota
WHERE
    FT.AnoMesRef > '202311'
"""

# Executa a consulta
print("📥 Executando a consulta SQL...")
df = pd.read_sql(query_table, conn)

# Converte colunas de data
df['DT_EntregaBem'] = pd.to_datetime(df['DT_EntregaBem'], errors='coerce')
df['DT_Contemplacao'] = pd.to_datetime(df['DT_Contemplacao'], errors='coerce')

# Caminho de saída
pasta_destino = rf'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\TABELA Milton\Tabela Oficial'
excel_path = os.path.join(pasta_destino, 'matriz_tempo_entrega.xlsx')

# Adiciona coluna 'Safra Contemplacao' no formato YYYYMM
df['Safra Contemplacao'] = df['DT_Contemplacao'].dt.strftime('%Y%m')

# Mapeamento de categorias para colunas M1...M14
mapa_categorias = {
    'Over 30': 'M1',
    'Over 60': 'M2',
    'Over 90': 'M3',
    'Over 120': 'M4',
    'Over 150': 'M5',
    'Over 180': 'M6',
    'Over 210': 'M7',
    'Over 240': 'M8',
    'Over 270': 'M9',
    'Over 300': 'M10',
    'Over 330': 'M11',
    'Over 360': 'M12',
    'Over 390': 'M13',
    'Over > 390': 'M14'
}
df['Categoria_M'] = df['Categoria_Tempo_Entrega'].map(mapa_categorias)

# Remove linhas inválidas
df_valid = df.dropna(subset=['Categoria_M', 'Safra Contemplacao'])

# Descobre dinamicamente as colunas M presentes nos dados
colunas_m = sorted(df_valid['Categoria_M'].unique(), key=lambda x: int(x[1:]))

# ===== Matriz Por Cotas =====
matriz_cotas = (
    df_valid.groupby(['Safra Contemplacao', 'Categoria_M'])['ID_Cota']
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
    .rename_axis(None, axis=1)
)
matriz_cotas = matriz_cotas.reindex(columns=['Safra Contemplacao'] + colunas_m)
matriz_cotas = matriz_cotas[matriz_cotas['Safra Contemplacao'] >= '202403']

# ===== Matriz Por Clientes =====
matriz_clientes = (
    df_valid.groupby(['Safra Contemplacao', 'Categoria_M'])['CD_InscricaoNacional']
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
    .rename_axis(None, axis=1)
)
matriz_clientes = matriz_clientes.reindex(columns=['Safra Contemplacao'] + colunas_m)
matriz_clientes = matriz_clientes[matriz_clientes['Safra Contemplacao'] >= '202403']

# ===== Exporta para Excel com formatação =====
with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    matriz_cotas.to_excel(writer, sheet_name='Geral Por Cotas Distintas', index=False)
    matriz_clientes.to_excel(writer, sheet_name='Geral Por Clientes', index=False)

    workbook = writer.book
    alinhamento_esquerda = workbook.add_format({'align': 'left'})

    def formatar_planilha(writer, sheet_name, df):
        worksheet = writer.sheets[sheet_name]
        for idx, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 1
            worksheet.set_column(idx, idx, max_len, alinhamento_esquerda)

        # Aplica heatmap em todas as colunas numéricas
        for idx, col in enumerate(df.columns[1:], start=1):
            worksheet.conditional_format(
                1, idx, len(df), idx,
                {
                    'type': '3_color_scale',
                    'min_color': "#63BE7B",
                    'mid_color': "#FFEB84",
                    'max_color': "#F8696B"
                }
            )

    formatar_planilha(writer, 'Geral Por Cotas Distintas', matriz_cotas)
    formatar_planilha(writer, 'Geral Por Clientes', matriz_clientes)

print(f"✅ Excel final exportado com heatmap, colunas ajustadas:\n{excel_path}")


📥 Executando a consulta SQL...
✅ Excel final exportado com heatmap, colunas ajustadas:
C:\Users\GabrielHenriqueSilva\OneDrive - CAIXA Consórcio\Documentos\TABELA Milton\Tabela Oficial\matriz_tempo_entrega.xlsx
